# Analyse exploratoire — lesfourcasters

**Projet** : impact sanitaire des vagues de chaleur en France
**Autrice** : Angel Thevenet — Wild Code School, promotion mai 2026
**Source météo** : Open-Meteo (réanalyse ERA5), 360 communes, 2000-2026
**Source santé** : Odissé, Santé publique France, maille départementale

---

## Objectif de ce notebook

Ce notebook documente l'analyse exploratoire menée sur les données du projet avant la phase
de modélisation. Il poursuit trois buts :

1. **Vérifier la qualité des données** produites par le pipeline dbt — complétude, cohérence,
   unicité, continuité temporelle
2. **Comprendre la structure** des variables météorologiques et sanitaires
3. **Identifier les signaux exploitables** pour le modèle de prédiction de la surmortalité

Chaque contrôle de qualité effectué ici a été intégré au pipeline sous forme de test dbt
ou de correction dans les modèles de transformation.

---

## Prérequis

```bash
pip install google-cloud-bigquery pandas matplotlib seaborn db-dtypes
gcloud auth application-default login
```

L'authentification utilise les credentials par défaut de l'environnement local
(`gcloud auth application-default login`), les mêmes que ceux du profil dbt `dev`.

## 1. Connexion et configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from google.cloud import bigquery

PROJECT = 'newfourcasters'
DATASET_RAW = 'lesfourcasters_raw'
DATASET_DBT = 'lesfourcasters_dbt'

client = bigquery.Client(project=PROJECT)


def q(sql):
    """Execute une requete BigQuery et renvoie un DataFrame."""
    return client.query(sql).to_dataframe()


print(f"Connecte au projet {PROJECT}")

In [ ]:
# Palette du projet (identite visuelle terracotta)
TERRACOTTA = '#A8462A'
TERRA_CLAIR = '#C86B47'
SABLE = '#E3C4B4'
NAVY = '#2C3E50'
VERT = '#4A7C59'

PALETTE = [TERRACOTTA, NAVY, TERRA_CLAIR, VERT, SABLE]

sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.figsize': (11, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.edgecolor': '#666666',
    'grid.color': '#E8E8E8',
    'font.size': 10,
})

## 2. Inventaire des sources

Le pipeline alimente deux zones dans BigQuery : `lesfourcasters_raw` pour les données brutes
telles que collectées, et `lesfourcasters_dbt` pour les tables transformées par dbt.

In [ ]:
tables = []
for ds in (DATASET_RAW, DATASET_DBT):
    for t in client.list_tables(f"{PROJECT}.{ds}"):
        ref = client.get_table(f"{PROJECT}.{ds}.{t.table_id}")
        tables.append({
            'dataset': ds,
            'table': t.table_id,
            'lignes': ref.num_rows,
            'colonnes': len(ref.schema),
            'taille_Mo': round(ref.num_bytes / 1024**2, 1),
        })

inventaire = pd.DataFrame(tables).sort_values(['dataset', 'lignes'], ascending=[True, False])
inventaire

### Couverture de la table de faits

`fct_daily_heat_health` est la table finale du pipeline : une ligne par commune et par jour,
enrichie du code INSEE, du département et de la région.

In [ ]:
couverture = q(f"""
SELECT
  COUNT(*)                          AS total_lignes,
  COUNT(DISTINCT ville)             AS nb_communes,
  COUNT(DISTINCT code_insee)        AS nb_codes_insee,
  COUNT(DISTINCT departement)       AS nb_departements,
  COUNT(DISTINCT region)            AS nb_regions,
  MIN(DATE(date))                   AS date_min,
  MAX(DATE(date))                   AS date_max,
  COUNT(DISTINCT DATE(date))        AS nb_jours
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
""")

couverture.T.rename(columns={0: 'valeur'})

**Point de contrôle.** Le nombre de communes doit être égal au nombre de codes INSEE.
Un écart signale des variantes d'orthographe pour une même commune — c'est exactement ce que
nous avons détecté lors de l'audit (voir section 4).

## 3. Contrôles de qualité

Cette section reprend les contrôles menés sur le pipeline. Chacun a donné lieu soit à une
correction dans les modèles dbt, soit à un test de non-régression.

### 3.1 Valeurs manquantes

In [ ]:
nulls = q(f"""
SELECT
  COUNT(*)                                  AS total,
  COUNTIF(date IS NULL)                     AS date_null,
  COUNTIF(ville IS NULL)                    AS ville_null,
  COUNTIF(code_insee IS NULL)               AS code_insee_null,
  COUNTIF(departement IS NULL)              AS departement_null,
  COUNTIF(region IS NULL)                   AS region_null,
  COUNTIF(temperature_moyenne IS NULL)      AS temp_moyenne_null,
  COUNTIF(humidite_moyenne IS NULL)         AS humidite_null,
  COUNTIF(vitesse_vent_moyenne IS NULL)     AS vent_null,
  COUNTIF(pression_moyenne IS NULL)         AS pression_null
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
""")

total = nulls['total'].iloc[0]
resume = nulls.drop(columns='total').T.rename(columns={0: 'nb_nulls'})
resume['taux'] = (resume['nb_nulls'] / total * 100).round(3).astype(str) + ' %'
resume

La table de faits ne contient aucune valeur manquante. Ce résultat n'était pas acquis :
la source brute présentait initialement plus de 870 000 lignes sans nom de commune ni
coordonnées, soit environ un quart du volume.

Deux transformations dbt ont permis de les récupérer :

- `IFNULL(Ville, nom_poi)` dans `stg_open_meteo` — le fichier source alimente tantôt une colonne,
  tantôt l'autre
- une double jointure dans `int_health_weather_join` — d'abord sur le numéro de département,
  puis en repli sur le nom de commune normalisé

In [ ]:
# Verification du taux de completude sur la couche brute, avant transformation
nulls_raw = q(f"""
SELECT
  COUNT(*)                              AS total,
  COUNTIF(nom_poi IS NULL)              AS nom_poi_null,
  COUNTIF(latitude_poi IS NULL)         AS latitude_null,
  COUNTIF(temperature_2m_mean IS NULL)  AS temp_null
FROM `{PROJECT}.{DATASET_RAW}.raw_open_meteo`
""")

t = nulls_raw['total'].iloc[0]
print(f"Table brute : {t:,} lignes".replace(',', ' '))
for col in ['nom_poi_null', 'latitude_null', 'temp_null']:
    n = nulls_raw[col].iloc[0]
    print(f"  {col:20} {n:>10,}  ({n/t*100:5.2f} %)".replace(',', ' '))

### 3.2 Bornes physiques des variables

In [ ]:
bornes = q(f"""
SELECT
  ROUND(MIN(temperature_minimale), 1)   AS temp_min_absolue,
  ROUND(MAX(temperature_maximale), 1)   AS temp_max_absolue,
  ROUND(MIN(humidite_moyenne), 1)       AS humidite_min,
  ROUND(MAX(humidite_moyenne), 1)       AS humidite_max,
  ROUND(MIN(precipitations_totales), 1) AS precip_min,
  ROUND(MAX(precipitations_totales), 1) AS precip_max,
  ROUND(MAX(vitesse_vent_maximale), 1)  AS vent_max,
  ROUND(MIN(pression_moyenne), 1)       AS pression_min,
  ROUND(MAX(pression_moyenne), 1)       AS pression_max
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
""")

bornes.T.rename(columns={0: 'valeur'})

**Interprétation.** Toutes les valeurs se situent dans des plages physiquement plausibles
pour la France métropolitaine : températures entre -28 °C et +44 °C, humidité relative bornée
à 100 %, précipitations positives, pression atmosphérique autour de 1013 hPa.

Aucune valeur sentinelle (-999, -9999) n'a été détectée. C'est un contrôle important : les NULLs
ne sont pas le seul type d'anomalie, et une valeur aberrante passe silencieusement dans un
modèle de machine learning.

### 3.3 Cohérence interne

In [ ]:
coherence = q(f"""
SELECT
  COUNTIF(temperature_minimale > temperature_maximale)          AS min_superieur_max,
  COUNTIF(temperature_moyenne < temperature_minimale)           AS moyenne_sous_min,
  COUNTIF(temperature_moyenne > temperature_maximale)           AS moyenne_sur_max,
  COUNTIF(humidite_moyenne < 0 OR humidite_moyenne > 100)       AS humidite_hors_bornes,
  COUNTIF(precipitations_totales < 0)                           AS precipitations_negatives,
  COUNTIF(pluie_totale > precipitations_totales)                AS pluie_superieure_precip
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
""")

resultat = coherence.T.rename(columns={0: 'nb_anomalies'})
resultat['statut'] = np.where(resultat['nb_anomalies'] == 0, 'OK', 'A CORRIGER')
resultat

### 3.4 Unicité de la clé métier et continuité temporelle

In [ ]:
unicite = q(f"""
SELECT
  COUNT(*) AS total_lignes,
  COUNT(DISTINCT CONCAT(CAST(DATE(date) AS STRING), '|', ville)) AS couples_uniques
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
""")

n_tot = unicite['total_lignes'].iloc[0]
n_uniq = unicite['couples_uniques'].iloc[0]
print(f"Lignes             : {n_tot:,}".replace(',', ' '))
print(f"Couples uniques    : {n_uniq:,}".replace(',', ' '))
print(f"Doublons           : {n_tot - n_uniq:,}".replace(',', ' '))

In [ ]:
continuite = q(f"""
SELECT
  MIN(DATE(date))                                              AS date_min,
  MAX(DATE(date))                                              AS date_max,
  COUNT(DISTINCT DATE(date))                                   AS jours_presents,
  DATE_DIFF(MAX(DATE(date)), MIN(DATE(date)), DAY) + 1         AS jours_attendus
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
""")

presents = continuite['jours_presents'].iloc[0]
attendus = continuite['jours_attendus'].iloc[0]
print(f"Periode      : {continuite['date_min'].iloc[0]} -> {continuite['date_max'].iloc[0]}")
print(f"Jours presents / attendus : {presents} / {attendus}")
print(f"Jours manquants           : {attendus - presents}")

In [ ]:
# Verification plus stricte : toutes les communes sont-elles presentes chaque jour ?
lacunes = q(f"""
SELECT
  DATE(date) AS jour,
  COUNT(DISTINCT ville) AS nb_communes
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
GROUP BY jour
HAVING COUNT(DISTINCT ville) < 360
ORDER BY jour
""")

if lacunes.empty:
    print("Couverture complete : 360 communes presentes chaque jour.")
else:
    print(f"{len(lacunes)} jours incomplets :")
    display(lacunes.head(20))

## 4. Anomalies détectées et corrigées

Trois problèmes ont été identifiés pendant l'exploration et corrigés dans le pipeline.

### 4.1 Variantes d'orthographe des noms de communes

Le fichier météo historique et le référentiel communes n'écrivent pas les noms de la même
manière. Deux communes apparaissaient en double dans les analyses.

In [ ]:
variantes = q(f"""
SELECT
  code_insee,
  COUNT(DISTINCT ville) AS nb_variantes,
  STRING_AGG(DISTINCT ville, ' | ') AS orthographes
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
GROUP BY code_insee
HAVING COUNT(DISTINCT ville) > 1
""")

if variantes.empty:
    print("Aucune commune avec plusieurs orthographes : correction effective.")
else:
    display(variantes)

**Les cas rencontrés :**

| Code INSEE | Fichier météo | Référentiel |
|---|---|---|
| 04070 | `Digne-les-Bains` | `Digne-les-bains` |
| 48200 | `Saint-Chely-d'Apcher` | `Saint-Chely-d-Apcher ` (apostrophe remplacée, espace final) |

**La correction appliquée** dans `int_health_weather_join` normalise les deux côtés de la
jointure avant comparaison :

```sql
REGEXP_REPLACE(NORMALIZE(LOWER(TRIM(ville)), NFD), r"[\pM'’\-\s]", '')
```

`NORMALIZE(..., NFD)` décompose les caractères accentués, la classe `\pM` retire les marques
diacritiques, et le reste de la classe supprime apostrophes, tirets et espaces. Le nom
définitif est ensuite repris du référentiel, qui fait autorité.

### 4.2 Doublons créés par une clé de déduplication trop large

Le script de collecte calculait initialement son hash sur `date + commune + latitude + longitude`.
Les coordonnées étant stockées en flottant, leur représentation différait légèrement entre
l'API et BigQuery : le hash ne correspondait jamais, et chaque exécution réinsérait les mêmes
lignes.

La clé métier a été ramenée à `date + commune` — une commune n'a qu'une seule mesure par jour.

In [ ]:
# Trace des insertions : une execution du pipeline par jour
insertions = q(f"""
SELECT
  DATE(insere_a)        AS jour_insertion,
  COUNT(*)              AS lignes_inserees,
  MAX(DATE(time))       AS donnee_la_plus_recente
FROM `{PROJECT}.{DATASET_RAW}.raw_open_meteo`
WHERE insere_a IS NOT NULL
GROUP BY jour_insertion
ORDER BY jour_insertion DESC
""")

insertions

Après correction, chaque exécution quotidienne n'insère plus que 360 lignes — une par
commune — alors que la fenêtre glissante en interroge 3 600. La déduplication écarte
correctement les 3 240 déjà présentes.

### 4.3 Structure réelle des données Odissé

Le champ `libgeo` du jeu de données canicule était initialement interprété comme un nom de
commune. L'exploration montre qu'il contient en réalité des **départements**.

In [ ]:
odisse = q(f"""
SELECT
  departement_nom,
  departement_code,
  COUNT(*) AS nb_annees
FROM `{PROJECT}.{DATASET_DBT}.stg_odisse_canicule`
GROUP BY departement_nom, departement_code
ORDER BY departement_nom
LIMIT 10
""")

odisse

**Conséquence structurante pour la suite du projet.** Le croisement météo/santé ne peut pas
se faire à la commune : il faut agréger la météo quotidienne communale vers la maille
**département-année**, seule maille commune aux deux sources.

Les colonnes ont été renommées dans `stg_odisse_canicule` (`libgeo` → `departement_nom`,
`nb_j_can` → `nb_jours_canicule`) pour éviter toute ambiguïté ultérieure.

## 5. Distribution des variables météorologiques

L'échantillonnage porte sur les données depuis 2015 pour limiter le volume transféré.

In [ ]:
meteo = q(f"""
SELECT
  DATE(date)          AS date,
  ville,
  departement,
  region,
  code_insee,
  temperature_moyenne,
  temperature_minimale,
  temperature_maximale,
  humidite_moyenne,
  precipitations_totales,
  vitesse_vent_moyenne,
  pression_moyenne,
  duree_ensoleillement
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
WHERE DATE(date) >= '2015-01-01'
""")

meteo['date'] = pd.to_datetime(meteo['date'])
meteo['annee'] = meteo['date'].dt.year
meteo['mois'] = meteo['date'].dt.month

print(f"{len(meteo):,} lignes chargees".replace(',', ' '))
meteo.head()

In [ ]:
meteo.describe().T.round(2)

In [ ]:
variables = [
    ('temperature_moyenne', 'Température moyenne (°C)'),
    ('temperature_maximale', 'Température maximale (°C)'),
    ('humidite_moyenne', 'Humidité relative (%)'),
    ('vitesse_vent_moyenne', 'Vitesse du vent (km/h)'),
    ('pression_moyenne', 'Pression au niveau de la mer (hPa)'),
    ('duree_ensoleillement', 'Durée d\'ensoleillement (s)'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (col, titre) in zip(axes.ravel(), variables):
    ax.hist(meteo[col].dropna(), bins=60, color=TERRACOTTA, edgecolor='white', linewidth=0.3)
    ax.set_title(titre)
    ax.set_ylabel('')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{int(x/1000)}k' if x >= 1000 else int(x)))

fig.suptitle('Distribution des variables météorologiques (2015-2026)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

La distribution des températures est bimodale, ce qui traduit simplement l'alternance des
saisons. Les précipitations, elles, suivent une loi très asymétrique : la majorité des jours
sont secs, quelques jours concentrent des cumuls importants.

## 6. Saisonnalité et tendance de long terme

In [ ]:
mensuel = (meteo.groupby('mois')
           .agg(temp_moy=('temperature_moyenne', 'mean'),
                temp_max=('temperature_maximale', 'mean'),
                temp_min=('temperature_minimale', 'mean'))
           .reset_index())

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(mensuel['mois'], mensuel['temp_min'], mensuel['temp_max'],
                color=SABLE, alpha=0.6, label='Amplitude min-max')
ax.plot(mensuel['mois'], mensuel['temp_moy'], color=TERRACOTTA, linewidth=2.5, marker='o', label='Moyenne')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Juin',
                    'Juil', 'Août', 'Sep', 'Oct', 'Nov', 'Déc'])
ax.set_title('Cycle saisonnier des températures (moyenne 2015-2026, 360 communes)')
ax.set_ylabel('Température (°C)')
ax.legend()
plt.tight_layout()
plt.show()

### Tendance sur 26 ans

L'analyse porte ici sur l'ensemble de l'historique, agrégé côté BigQuery pour éviter de
rapatrier 3,5 millions de lignes.

In [ ]:
tendance = q(f"""
SELECT
  EXTRACT(YEAR FROM date)                    AS annee,
  ROUND(AVG(temperature_moyenne), 2)         AS temp_moyenne_annuelle,
  ROUND(AVG(temperature_maximale), 2)        AS temp_max_moyenne,
  ROUND(MAX(temperature_maximale), 1)        AS temp_max_absolue
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
GROUP BY annee
ORDER BY annee
""")

# 2026 est incomplete : on l'ecarte de la regression
complet = tendance[tendance['annee'] < 2026]
pente, ordonnee = np.polyfit(complet['annee'], complet['temp_moyenne_annuelle'], 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(tendance['annee'], tendance['temp_moyenne_annuelle'],
        color=TERRACOTTA, linewidth=2, marker='o', markersize=5, label='Température moyenne')
ax.plot(complet['annee'], pente * complet['annee'] + ordonnee,
        color=NAVY, linestyle='--', linewidth=1.8,
        label=f'Tendance : {pente*10:+.2f} °C par décennie')

ax.set_title('Évolution de la température moyenne annuelle (moyenne des 360 communes)')
ax.set_ylabel('Température (°C)')
ax.set_xlabel('')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Réchauffement estimé : {pente*10:+.2f} °C par décennie sur 2000-2025")

**Lecture.** La tendance est un ajustement linéaire simple, à interpréter avec prudence :
26 ans est une durée courte à l'échelle climatique, et la variabilité interannuelle est forte.
L'année 2026 est exclue du calcul car incomplète (données arrêtées en août).

## 7. Disparités géographiques

In [ ]:
par_dept = q(f"""
SELECT
  departement,
  region,
  ROUND(AVG(temperature_moyenne), 2)  AS temp_moyenne,
  ROUND(MAX(temperature_maximale), 1) AS temp_max_absolue,
  ROUND(AVG(humidite_moyenne), 1)     AS humidite_moyenne,
  COUNT(DISTINCT ville)               AS nb_communes
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
WHERE EXTRACT(YEAR FROM date) BETWEEN 2015 AND 2025
GROUP BY departement, region
ORDER BY temp_moyenne DESC
""")

print(f"{len(par_dept)} départements")
par_dept.head(15)

In [ ]:
top = par_dept.nlargest(15, 'temp_moyenne')
bottom = par_dept.nsmallest(15, 'temp_moyenne').iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].barh(top['departement'], top['temp_moyenne'], color=TERRACOTTA)
axes[0].set_title('15 départements les plus chauds')
axes[0].set_xlabel('Température moyenne (°C)')
axes[0].invert_yaxis()

axes[1].barh(bottom['departement'], bottom['temp_moyenne'], color=NAVY)
axes[1].set_title('15 départements les plus frais')
axes[1].set_xlabel('Température moyenne (°C)')
axes[1].invert_yaxis()

fig.suptitle('Température moyenne par département (2015-2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
par_region = (par_dept.groupby('region')
              .agg(temp_moyenne=('temp_moyenne', 'mean'),
                   temp_max=('temp_max_absolue', 'max'),
                   nb_depts=('departement', 'count'))
              .sort_values('temp_moyenne', ascending=True)
              .reset_index())

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(par_region['region'], par_region['temp_moyenne'], color=TERRA_CLAIR)
ax.set_title('Température moyenne par région (2015-2025)')
ax.set_xlabel('Température moyenne (°C)')
plt.tight_layout()
plt.show()

par_region

## 8. Identification des épisodes de forte chaleur

Une canicule au sens du système d'alerte de Santé publique France se définit par le dépassement
simultané de seuils départementaux sur les températures minimale et maximale, en moyenne
glissante sur trois jours. Ces seuils varient d'un département à l'autre.

Faute de disposer des seuils officiels, on approche ici la notion par un critère simple :
le nombre de jours où la température maximale dépasse 35 °C.

In [ ]:
chaleur = q(f"""
SELECT
  EXTRACT(YEAR FROM date) AS annee,
  COUNTIF(temperature_maximale > 30) AS jours_sup_30,
  COUNTIF(temperature_maximale > 35) AS jours_sup_35,
  COUNTIF(temperature_maximale > 40) AS jours_sup_40,
  COUNT(DISTINCT IF(temperature_maximale > 35, departement, NULL)) AS depts_touches_35
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
GROUP BY annee
ORDER BY annee
""")

chaleur

In [ ]:
complet_ch = chaleur[chaleur['annee'] < 2026]

fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.bar(complet_ch['annee'], complet_ch['jours_sup_35'],
        color=TERRACOTTA, alpha=0.85, label='Jours-commune > 35 °C')
ax1.set_ylabel('Jours-commune au-dessus de 35 °C', color=TERRACOTTA)
ax1.tick_params(axis='y', labelcolor=TERRACOTTA)

ax2 = ax1.twinx()
ax2.plot(complet_ch['annee'], complet_ch['depts_touches_35'],
         color=NAVY, linewidth=2, marker='o', label='Départements touchés')
ax2.set_ylabel('Nombre de départements touchés', color=NAVY)
ax2.tick_params(axis='y', labelcolor=NAVY)
ax2.grid(False)

ax1.set_title('Intensité et étendue des épisodes de forte chaleur par année')
plt.tight_layout()
plt.show()

**Deux dimensions du risque.** Le nombre de jours-commune mesure l'intensité cumulée,
le nombre de départements touchés mesure l'étendue géographique. Un été peut être intense
et localisé, ou modéré et généralisé — les conséquences sanitaires et opérationnelles
diffèrent.

In [ ]:
# Repartition mensuelle des jours de forte chaleur
saison_chaleur = q(f"""
SELECT
  EXTRACT(MONTH FROM date) AS mois,
  COUNTIF(temperature_maximale > 35) AS jours_sup_35
FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
GROUP BY mois
ORDER BY mois
""")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(saison_chaleur['mois'], saison_chaleur['jours_sup_35'], color=TERRACOTTA)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Juin',
                    'Juil', 'Août', 'Sep', 'Oct', 'Nov', 'Déc'])
ax.set_title('Répartition mensuelle des jours-commune au-dessus de 35 °C (2000-2026)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 9. Corrélations entre variables météorologiques

In [ ]:
cols_num = [
    'temperature_moyenne', 'temperature_maximale', 'temperature_minimale',
    'humidite_moyenne', 'precipitations_totales',
    'vitesse_vent_moyenne', 'pression_moyenne', 'duree_ensoleillement',
]

corr = meteo[cols_num].corr()

fig, ax = plt.subplots(figsize=(9.5, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.75}, ax=ax)
ax.set_title('Matrice de corrélation des variables météorologiques', pad=14)
plt.tight_layout()
plt.show()

**À retenir pour la modélisation.** Les trois variables de température sont fortement
corrélées entre elles, ce qui est attendu. Cette colinéarité devra être traitée lors de la
sélection des variables du modèle : conserver les trois n'apporterait pas d'information
supplémentaire mais rendrait les coefficients instables.

La corrélation négative entre température et humidité est un signal physique classique.
L'ensoleillement et la température évoluent ensemble, sans redondance totale.

## 10. Premier croisement météo / santé

Les données sanitaires étant à la maille département-année, la météo doit être agrégée au
même niveau. Cette agrégation préfigure la table qui alimentera le modèle prédictif.

In [ ]:
croisement = q(f"""
WITH meteo_agregee AS (
  SELECT
    EXTRACT(YEAR FROM date)                    AS annee,
    departement,
    ROUND(AVG(temperature_moyenne), 2)         AS temp_moyenne_annuelle,
    ROUND(MAX(temperature_maximale), 1)        AS temp_max_absolue,
    COUNTIF(temperature_maximale > 35)         AS jours_sup_35,
    COUNTIF(temperature_maximale > 30)         AS jours_sup_30,
    ROUND(AVG(humidite_moyenne), 1)            AS humidite_moyenne,
    ROUND(SUM(precipitations_totales), 1)      AS precipitations_annuelles
  FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
  GROUP BY annee, departement
),

meteo_ete AS (
  SELECT
    EXTRACT(YEAR FROM date)              AS annee,
    departement,
    ROUND(AVG(temperature_moyenne), 2)   AS temp_moyenne_ete,
    ROUND(AVG(temperature_maximale), 2)  AS temp_max_moyenne_ete
  FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
  WHERE EXTRACT(MONTH FROM date) BETWEEN 6 AND 9
  GROUP BY annee, departement
)

SELECT
  m.annee,
  m.departement,
  m.temp_moyenne_annuelle,
  e.temp_moyenne_ete,
  e.temp_max_moyenne_ete,
  m.temp_max_absolue,
  m.jours_sup_30,
  m.jours_sup_35,
  m.humidite_moyenne,
  m.precipitations_annuelles,
  o.nb_jours_canicule
FROM meteo_agregee m
JOIN meteo_ete e
  ON m.annee = e.annee AND m.departement = e.departement
JOIN `{PROJECT}.{DATASET_DBT}.stg_odisse_canicule` o
  ON CAST(m.annee AS STRING) = CAST(EXTRACT(YEAR FROM o.annee) AS STRING)
 AND m.departement = o.departement_nom
ORDER BY m.annee DESC, m.departement
""")

print(f"{len(croisement)} observations département-année")
croisement.head(10)

Si la jointure ci-dessus renvoie peu de lignes, la cause la plus probable est un écart
d'orthographe entre les noms de départements des deux sources, ou un type incompatible sur
le champ année. La cellule suivante permet de diagnostiquer.

In [ ]:
# Diagnostic de la jointure : quels departements ne matchent pas ?
diag = q(f"""
WITH depts_meteo AS (
  SELECT DISTINCT departement FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
),
depts_odisse AS (
  SELECT DISTINCT departement_nom FROM `{PROJECT}.{DATASET_DBT}.stg_odisse_canicule`
)
SELECT
  (SELECT COUNT(*) FROM depts_meteo)                                    AS nb_depts_meteo,
  (SELECT COUNT(*) FROM depts_odisse)                                   AS nb_depts_odisse,
  (SELECT COUNT(*) FROM depts_meteo m
   JOIN depts_odisse o ON m.departement = o.departement_nom)            AS nb_correspondances
""")

diag.T.rename(columns={0: 'valeur'})

In [ ]:
# Departements presents cote meteo mais absents cote Odisse (et inversement)
ecarts = q(f"""
SELECT 'meteo seulement' AS source, departement AS nom
FROM (SELECT DISTINCT departement FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`)
WHERE departement NOT IN (
  SELECT DISTINCT departement_nom FROM `{PROJECT}.{DATASET_DBT}.stg_odisse_canicule`
)
UNION ALL
SELECT 'odisse seulement' AS source, departement_nom AS nom
FROM (SELECT DISTINCT departement_nom FROM `{PROJECT}.{DATASET_DBT}.stg_odisse_canicule`)
WHERE departement_nom NOT IN (
  SELECT DISTINCT departement FROM `{PROJECT}.{DATASET_DBT}.fct_daily_heat_health`
)
ORDER BY source, nom
""")

if ecarts.empty:
    print("Les deux référentiels de départements coïncident parfaitement.")
else:
    print(f"{len(ecarts)} écarts détectés :")
    display(ecarts)

### Relation entre température estivale et jours de canicule

In [ ]:
if not croisement.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

    axes[0].scatter(croisement['temp_max_moyenne_ete'], croisement['nb_jours_canicule'],
                    alpha=0.4, color=TERRACOTTA, s=22)
    axes[0].set_xlabel('Température maximale moyenne en été (°C)')
    axes[0].set_ylabel('Jours de canicule (Odissé)')
    axes[0].set_title('Canicule vs température estivale')

    axes[1].scatter(croisement['jours_sup_35'], croisement['nb_jours_canicule'],
                    alpha=0.4, color=NAVY, s=22)
    axes[1].set_xlabel('Jours-commune au-dessus de 35 °C')
    axes[1].set_ylabel('Jours de canicule (Odissé)')
    axes[1].set_title('Canicule vs jours de forte chaleur')

    fig.suptitle('Croisement des indicateurs météo et de l\'indicateur sanitaire',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    r1 = croisement['temp_max_moyenne_ete'].corr(croisement['nb_jours_canicule'])
    r2 = croisement['jours_sup_35'].corr(croisement['nb_jours_canicule'])
    print(f"Corrélation température estivale / jours de canicule : {r1:.3f}")
    print(f"Corrélation jours > 35 °C / jours de canicule        : {r2:.3f}")
else:
    print("Croisement vide : corriger la jointure avant de poursuivre.")

**Ce que cela indique pour le modèle.** Une corrélation forte entre les indicateurs météo
calculés depuis ERA5 et l'indicateur officiel de Santé publique France valide l'approche :
les variables dont nous disposons portent bien le signal du phénomène que nous cherchons
à modéliser.

Il faut cependant garder à l'esprit que l'indicateur Odissé est lui-même construit à partir
de températures — celles de Météo-France, sur une station de référence par département.
Une corrélation élevée est donc attendue et ne constitue pas en soi une découverte. Le véritable
enjeu du modèle sera de prédire non pas les jours de canicule, mais **la surmortalité associée**,
qui dépend aussi de facteurs démographiques et sociaux.

## 11. Synthèse et suites

### Qualité des données

| Contrôle | Résultat |
|---|---|
| Valeurs manquantes sur les colonnes clés | Aucune |
| Bornes physiques des variables | Toutes plausibles |
| Cohérence interne (min ≤ moy ≤ max) | Aucune anomalie |
| Unicité de la clé (date, commune) | Respectée |
| Continuité de la série temporelle | Aucun jour manquant |
| Couverture géographique | 360 communes présentes chaque jour |

### Corrections apportées au pipeline

1. Récupération des noms de communes manquants par `IFNULL` et double jointure
2. Normalisation des libellés de communes (accents, apostrophes, espaces) avant jointure
3. Correction de la clé de déduplication, ramenée à `date + commune`
4. Renommage des colonnes Odissé après identification de la maille réelle
5. Collecte des 29 variables ERA5 au lieu des 4 initiales

### Ce que l'exploration apprend pour la modélisation

- Les variables de température sont fortement colinéaires : une sélection sera nécessaire
- Les phénomènes de forte chaleur se concentrent sur juin-septembre, ce qui justifie de
  calculer des agrégats saisonniers plutôt qu'annuels
- Les disparités départementales sont marquées : la localisation devra figurer parmi les
  variables du modèle
- La maille de travail pour le croisement santé est le couple département-année

### Étapes suivantes

1. Collecter les cinq jeux de données Odissé par API, dont les décès attribuables à la chaleur
   qui constitueront la cible du modèle
2. Construire un modèle dbt d'agrégation météo à la maille département-année
3. Entraîner un modèle de régression et évaluer ses performances
4. Restituer dans un dashboard Power BI et une application Streamlit